# 金融期货半自动更新

每次下载完成后，直接 **从头运行全部单元格（Run All）**。没有 ZIP 时先规划；有 ZIP 时先校验入库，成功复核后删除该 ZIP，再规划下一批。无需手改日期、验收范围或切换命令行。

你只需手动完成两件事：把最后一格代码复制到聚宽研究 Notebook 执行；下载成功发布的 `JQ_FINANCIAL_FUTURES_TRANSFER.zip` 并覆盖本项目 `data/inbox/` 同名文件，然后再次 Run All。不要提前放入仍在下载、没有本次下载指令或来源不明的文件。

最后一格显示本批聚宽代码和预计还需的完整下载轮数（包含本批），或说明更新完成及明天北京时间 00:01 后可以再次运行。来源确认缺失不冒充行情齐全；日历水位落后也不冒充已更新到最新。

本地只写本项目 DuckDB，不写正式湖、不自动登录聚宽、不自动下载。任何校验、入库或清理失败都会停止本轮并保留证据。旧版本 Notebook 和输出已归档，当前从头运行会重新生成全部结果。


In [1]:
import pathlib
import importlib
import sys
import datetime
import time
import json
import hashlib
import base64
import zlib
import html
from zoneinfo import ZoneInfo

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "02_Futures_Lakehouse"))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings
from config.data_contracts import TRADE_CALENDAR_SCHEMA, FUTURES_BAR_CALENDAR_SCHEMA
from a00_03_notebook_schema_browser import display_schema_metadata

financial_futures_dir = candidate_root / "01_project_collection" / "JQ_strategy" / "financial_futures_data"
sys.path.insert(0, str(financial_futures_dir))
import pandas as pd
from IPython.display import display, Markdown
import financial_futures_collection_policy as collection_policy
import financial_futures_local_contract as local_contract
import financial_futures_fetch_planner as fetch_planner_module
import import_financial_futures_file as file_import_module
for workflow_module in (collection_policy, local_contract, fetch_planner_module, file_import_module):
    importlib.reload(workflow_module)
from financial_futures_fetch_planner import build_financial_futures_fetch_plan, FETCH_PLANNER_VERSION
from import_financial_futures_file import import_financial_futures_file

# 每次从头运行重新建立本轮状态，不允许旧内核中的计划绕过本轮入库。
fetch_plan = None
joinquant_code = None
import_report = None
round_ready_for_planning = False

print("Python:", sys.executable)
if pathlib.Path(sys.prefix).name.lower() != "latitude":
    raise RuntimeError("请选择 latitude 内核后从头运行，不要使用 base 环境。")
print("正式湖（只读）:", settings.futures_lake_root)
print("项目数据库:", financial_futures_dir / "data" / "warehouse" / "financial_futures.duckdb")


Python: E:\anaconda3\envs\latitude\python.exe
正式湖（只读）: E:\Latitude_Analytics_v2\03_Futures_Database\futures_lake
项目数据库: E:\Latitude_Analytics_v2\01_project_collection\JQ_strategy\financial_futures_data\data\warehouse\financial_futures.duckdb


## 1．上游 Schema 契约

直接展示权威 Schema，只有结构和开休市证据参与本项目范围；正式商品采集的完成状态不作为本项目状态。


In [2]:
display_schema_metadata([TRADE_CALENDAR_SCHEMA, FUTURES_BAR_CALENDAR_SCHEMA])


英文表名,中文表名,字段数,主键,Hive 分区,Schema 版本
dim_trade_calendar,中国期货交易日历维度表,11,calendar_date,year,1.1.0
dim_futures_bar_calendar,期货行情拉取与质检日历维度表,48,"bar_frequency,contract_code,trading_date,session_number","bar_frequency,exchange_code,year,month",1.4.0


## 2．项目库契约与范围

四表定义来自可执行契约，下方只展示，不创建表或连接数据库。白名单、就绪时点、批次上限也直接读取当前配置。


In [3]:
for table_name, columns in local_contract.DATABASE_TABLE_COLUMNS.items():
    display(Markdown("### " + table_name))
    display(pd.DataFrame(columns, columns=["field", "duckdb_type", "nullable"]))

print("交易所:", collection_policy.FINANCIAL_FUTURES_EXCHANGE_CODE)
print("品种:", sorted(collection_policy.FINANCIAL_FUTURES_UNDERLYING_CODES))
print("频率:", collection_policy.FINANCIAL_FUTURES_BAR_FREQUENCIES)
print("数据就绪:", collection_policy.JOINQUANT_MARKET_TIME_ZONE,
      "T+1", collection_policy.JOINQUANT_COMPLETED_DATA_READY_TIME)
print("单来源请求理论键上限:", collection_policy.JOINQUANT_MAX_EXPECTED_KEYS_PER_SOURCE_REQUEST)
print("正常规划单文件理论键上限:",
      collection_policy.JOINQUANT_MAX_PLANNED_EXPECTED_KEYS_PER_TRANSFER)
print("传输协议理论键硬上限 / 观察块 / 来源请求上限:",
      collection_policy.JOINQUANT_MAX_EXPECTED_KEYS_PER_TRANSFER,
      collection_policy.JOINQUANT_MAX_OBSERVATIONS_PER_TRANSFER,
      collection_policy.JOINQUANT_MAX_SOURCE_REQUESTS_PER_TRANSFER)


### ingest_batch

,field,duckdb_type,nullable
0,file_sha256,VARCHAR,False
1,run_id,VARCHAR,False
2,plan_sha256,VARCHAR,False
3,protocol_version,VARCHAR,False
4,generator_version,VARCHAR,False
5,fixed_file_name,VARCHAR,False
6,file_bytes,BIGINT,False
7,manifest_sha256,VARCHAR,False
8,publication_status,VARCHAR,False
9,run_started_at,TIMESTAMPTZ,False


### fetch_observation

,field,duckdb_type,nullable
0,observation_id,VARCHAR,False
1,file_sha256,VARCHAR,False
2,source_request_id,VARCHAR,False
3,source_request_ordinal,INTEGER,False
4,request_id,VARCHAR,False
5,request_ordinal,INTEGER,False
6,contract_code,VARCHAR,False
7,bar_frequency,VARCHAR,False
8,trading_date,DATE,False
9,session_number,TINYINT,False


### futures_daily

,field,duckdb_type,nullable
0,contract_code,VARCHAR,False
1,trading_date,DATE,False
2,previous_close,DOUBLE,True
3,open,DOUBLE,False
4,high,DOUBLE,False
5,low,DOUBLE,False
6,close,DOUBLE,False
7,volume,DOUBLE,False
8,money,DOUBLE,False
9,open_interest,DOUBLE,False


### futures_minute

,field,duckdb_type,nullable
0,contract_code,VARCHAR,False
1,bar_at,TIMESTAMPTZ,False
2,trading_date,DATE,False
3,session_number,TINYINT,False
4,open,DOUBLE,False
5,high,DOUBLE,False
6,low,DOUBLE,False
7,close,DOUBLE,False
8,volume,DOUBLE,False
9,money,DOUBLE,False


交易所: CCFX
品种: ['IC', 'IF', 'IH', 'IM', 'T', 'TF', 'TL', 'TS']
频率: ('1d', '1m')
数据就绪: Asia/Shanghai T+1 00:01:00
单来源请求理论键上限: 38000
正常规划单文件理论键上限: 250000
传输协议理论键硬上限 / 观察块 / 来源请求上限: 500000 50000 64


## 3．自动消化本地 ZIP

本格自动检查固定 inbox：没有文件就继续规划；有文件则直接校验、入库并删除已消化文件。相同文件曾入库时先完整复核，再删除，不重复写行情。删除只发生在数据库成功复核之后；只清理已确认属于本次的文件，不删除并发替换的新下载文件。

结果包含本次实际新增行数、是否删除 ZIP 和执行时间。失败时停止，不自动重试或继续生成代码。


In [4]:
round_ready_for_planning = False
fetch_plan = None
joinquant_code = None
import_report = None
inbox_path = financial_futures_dir / "data" / "inbox" / local_contract.TRANSFER_FIXED_FILE_NAME
if inbox_path.exists() or inbox_path.is_symlink():
    import_report = import_financial_futures_file(write=True)
    print(json.dumps(import_report, ensure_ascii=False, indent=2, sort_keys=True))
    if not import_report["inbox_deleted"] or inbox_path.exists():
        raise RuntimeError("入口还有未消化文件或文件未完成清理；请核查后重新从头运行，不生成新批次。")
    print("ZIP 已校验入库并删除；现在自动规划下一批。")
else:
    print("没有待消化的 ZIP；直接检查数据库并规划。")
round_ready_for_planning = True


没有待消化的 ZIP；直接检查数据库并规划。


## 4．全白名单缺失检测与本批规划

自动采用当前北京时间、既定品种白名单和完整正式日历，不再限定两个验收样本。只有达到就绪时间且尚无有效行情/成功观察覆盖的键进入普通待拉取。

同时核对上游自然日日历和各品种日线/分钟结构的最新日期；上游未更新时明确提示，仍可拉取本地已知范围，不会把缺少上游日期解释为全量完成。


In [5]:
if not round_ready_for_planning:
    raise RuntimeError("请从头运行 Notebook，先完成本轮 ZIP 检查。")
fetch_plan = None
joinquant_code = None
planning_started = time.perf_counter()
fetch_plan = build_financial_futures_fetch_plan()
print("模式：全白名单自动分批")
print("评估时点:", fetch_plan.as_of.isoformat())
print("数据就绪截止自然日:", fetch_plan.latest_due_trading_date)
print("上游已知最新应有交易日:", fetch_plan.latest_due_market_trading_date)
print("上游日历已跟上就绪日期:", fetch_plan.calendar_is_current)
print("项目数据库已存在:", fetch_plan.database_exists)
print("耗时（秒）:", round(time.perf_counter() - planning_started, 3))
print("计划 SHA-256:", fetch_plan.plan_sha256)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

模式：全白名单自动分批
评估时点: 2026-09-05T06:59:26.767824+08:00
数据就绪截止自然日: 2026-09-04
上游已知最新应有交易日: 2026-08-28
上游日历已跟上就绪日期: False
项目数据库已存在: True
耗时（秒）: 19.941
计划 SHA-256: 0545ab5dab521b366943f63412d293656e521e666255961a696e8032a39abc11


## 5．覆盖总览与尚未就绪范围

`data_missing` 是全部有效行情缺口；`source_confirmed_missing` 是已完整请求但来源缺失的部分；只有 `fetch_pending` 触发正常拉取。来源确认缺失持续可见，但不日常无限重拉。


In [6]:
display(fetch_plan.coverage_summary_df)
print("尚未就绪块数:", len(fetch_plan.not_yet_fetchable_blocks_df))
display(fetch_plan.not_yet_fetchable_blocks_df.head(30))

if not fetch_plan.calendar_is_current:
    print("注意：上游日历水位尚未跟上；完成提示不会宣称数据库已更新到最新。")
    display(fetch_plan.calendar_coverage_df)


,bar_frequency,expected_due_block_count,expected_due_key_count,present_key_count,data_missing_key_count,source_confirmed_missing_key_count,fetch_pending_key_count,not_yet_fetchable_block_count,planned_observation_count,planned_expected_key_count
0,1d,68654,68654,68120,534,534,0,0,0,0
1,1m,137308,17248050,17103870,144180,144180,0,0,0,0


尚未就绪块数: 0


,bar_frequency,contract_code,exchange_code,underlying_code,trading_date,session_number,session_start_at,session_end_at,schedule_status,evidence_level,expected_bar_count


注意：上游日历水位尚未跟上；完成提示不会宣称数据库已更新到最新。


,bar_frequency,underlying_code,latest_calendar_trading_date,is_current
0,1d,IC,2026-08-28,True
1,1d,IF,2026-08-28,True
2,1d,IH,2026-08-28,True
3,1d,IM,2026-08-28,True
4,1d,T,2026-08-28,True
5,1d,TF,2026-08-28,True
6,1d,TL,2026-08-28,True
7,1d,TS,2026-08-28,True
8,1m,IC,2026-08-28,True
9,1m,IF,2026-08-28,True


## 6．缺失品种／合约名单

下表的首尾日期只是缺失块的**包围范围**，不代表期间每个键都缺失。精确日线键与分钟连续缺段见下一格；完整表保留在变量里，预览不是总量。


In [7]:
missing_contract_summary_df = (
    fetch_plan.missing_blocks_df.groupby(
        ["underlying_code", "contract_code", "bar_frequency"], observed=True
    ).agg(
        first_missing_date=("trading_date", "min"),
        last_missing_date=("trading_date", "max"),
        missing_block_count=("trading_date", "size"),
        data_missing_key_count=("data_missing_key_count", "sum"),
        source_confirmed_missing_key_count=("source_confirmed_missing_key_count", "sum"),
        fetch_pending_key_count=("fetch_pending_key_count", "sum"),
    ).reset_index()
)
print("缺失合约—频率组合数:", len(missing_contract_summary_df))
with pd.option_context("display.max_columns", None):
    display(missing_contract_summary_df.head(40))


缺失合约—频率组合数: 6


,underlying_code,contract_code,bar_frequency,first_missing_date,last_missing_date,missing_block_count,data_missing_key_count,source_confirmed_missing_key_count,fetch_pending_key_count
0,TF,TF1303.CCFX,1d,2012-06-11,2013-03-08,181,181,181,0
1,TF,TF1303.CCFX,1m,2012-06-11,2013-03-08,362,48870,48870,0
2,TF,TF1306.CCFX,1d,2012-09-17,2013-06-14,174,174,174,0
3,TF,TF1306.CCFX,1m,2012-09-17,2013-06-14,348,46980,46980,0
4,TF,TF1309.CCFX,1d,2012-12-17,2013-09-13,179,179,179,0
5,TF,TF1309.CCFX,1m,2012-12-17,2013-09-13,358,48330,48330,0


## 7．精确缺失日期与分钟范围

可填写 `detail_contract_code` 后只重跑此格来查看某一合约，不改变拉取范围。
分钟范围两端均包含；每条只覆盖一个交易日、一个 Session、同一种缺失状态内真正连续的分钟。


In [8]:
detail_contract_code = None  # 例如 "IF2409.CCFX"
preview_rows = 40

daily_missing_df = fetch_plan.missing_blocks_df.loc[
    fetch_plan.missing_blocks_df["bar_frequency"].eq("1d")
].copy()
minute_missing_ranges_df = fetch_plan.minute_missing_ranges_df
if detail_contract_code is not None:
    daily_missing_df = daily_missing_df.loc[daily_missing_df["contract_code"].eq(detail_contract_code)]
    minute_missing_ranges_df = minute_missing_ranges_df.loc[
        minute_missing_ranges_df["contract_code"].eq(detail_contract_code)
    ]
print("日线精确缺失键数:", len(daily_missing_df))
display(daily_missing_df[["contract_code", "trading_date", "missing_state"]].head(preview_rows))
print("分钟连续缺段数:", len(minute_missing_ranges_df))
with pd.option_context("display.max_columns", None):
    display(minute_missing_ranges_df.head(preview_rows))


日线精确缺失键数: 534


,contract_code,trading_date,missing_state
50372,TF1303.CCFX,2012-06-11,source_confirmed_missing
50373,TF1303.CCFX,2012-06-12,source_confirmed_missing
50374,TF1303.CCFX,2012-06-13,source_confirmed_missing
50375,TF1303.CCFX,2012-06-14,source_confirmed_missing
50376,TF1303.CCFX,2012-06-15,source_confirmed_missing
50377,TF1303.CCFX,2012-06-18,source_confirmed_missing
50378,TF1303.CCFX,2012-06-19,source_confirmed_missing
50379,TF1303.CCFX,2012-06-20,source_confirmed_missing
50380,TF1303.CCFX,2012-06-21,source_confirmed_missing
50381,TF1303.CCFX,2012-06-25,source_confirmed_missing


分钟连续缺段数: 1068


,contract_code,trading_date,session_number,range_start_at,range_end_at,missing_key_count,missing_state
0,TF1303.CCFX,2012-06-11,1,2012-06-11 09:16:00+08:00,2012-06-11 11:30:00+08:00,135,source_confirmed_missing
1,TF1303.CCFX,2012-06-11,2,2012-06-11 13:01:00+08:00,2012-06-11 15:15:00+08:00,135,source_confirmed_missing
2,TF1303.CCFX,2012-06-12,1,2012-06-12 09:16:00+08:00,2012-06-12 11:30:00+08:00,135,source_confirmed_missing
3,TF1303.CCFX,2012-06-12,2,2012-06-12 13:01:00+08:00,2012-06-12 15:15:00+08:00,135,source_confirmed_missing
4,TF1303.CCFX,2012-06-13,1,2012-06-13 09:16:00+08:00,2012-06-13 11:30:00+08:00,135,source_confirmed_missing
5,TF1303.CCFX,2012-06-13,2,2012-06-13 13:01:00+08:00,2012-06-13 15:15:00+08:00,135,source_confirmed_missing
6,TF1303.CCFX,2012-06-14,1,2012-06-14 09:16:00+08:00,2012-06-14 11:30:00+08:00,135,source_confirmed_missing
7,TF1303.CCFX,2012-06-14,2,2012-06-14 13:01:00+08:00,2012-06-14 15:15:00+08:00,135,source_confirmed_missing
8,TF1303.CCFX,2012-06-15,1,2012-06-15 09:16:00+08:00,2012-06-15 11:30:00+08:00,135,source_confirmed_missing
9,TF1303.CCFX,2012-06-15,2,2012-06-15 13:01:00+08:00,2012-06-15 15:15:00+08:00,135,source_confirmed_missing


## 8．本批范围与剩余轮数

一次“拉取轮次”指复制代码到聚宽运行、下载一个 ZIP，再从头运行本 Notebook。一次代码里可有多个 API 请求，两种次数分别显示。

规划器用同一套顺序及容量限制排完当前待办，第一包生成本批代码，后续包用于预计剩余轮数；不是把行数简单除以上限。估计假定后续请求成功、当前日历和白名单不变，失败或日期增长后自动重算。分钟最小观察块是完整交易时段，不切碎 Session。


In [9]:
planned_source_requests_df = fetch_plan.planned_source_requests_df
print("本批来源请求数:", len(planned_source_requests_df))
print("本批观察块数:", len(fetch_plan.planned_observations_df))
print("本批理论键数:", int(planned_source_requests_df["expected_key_count"].sum()))
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(planned_source_requests_df.drop(columns=["source_request_id"]))

print("预计还需拉取轮数（含本轮）:", fetch_plan.remaining_transfer_count)
print("本轮成功消化后预计剩余轮数:", max(0, fetch_plan.remaining_transfer_count - 1))
print("当前待办合计 API 请求数:", fetch_plan.remaining_source_request_count)
display(fetch_plan.transfer_schedule_df)


本批来源请求数: 0
本批观察块数: 0
本批理论键数: 0


,source_request_ordinal,contract_code,bar_frequency,source_frequency,source_start_at,source_end_at,observation_count,expected_key_count


预计还需拉取轮数（含本轮）: 0
本轮成功消化后预计剩余轮数: 0
当前待办合计 API 请求数: 0


,transfer_number,source_request_count,observation_count,expected_key_count


## 9．聚宽导出模板（只定义文本，不在本地执行）

模板属于此 Notebook。它兼容聚宽实测的 Python 3.6 / Pandas 0.23 环境，只使用显式 `jqresearch.api` 和 `py_builtins`。此格较长，通常保持折叠即可。


In [10]:
joinquant_export_runtime = r'''
# 在聚宽研究 Notebook 的干净内核中，整段粘贴到一个单元格执行。
# 本段只使用内嵌计划；不读取本地磁盘、账号或凭据，不做业务自动重试。
def _run_jq_financial_futures_export():
    import builtins as py_builtins
    import base64
    import datetime
    import hashlib
    import io
    import json
    import math
    import numbers
    import os
    import platform
    import re
    import tempfile
    import time
    import uuid
    import warnings
    import zipfile
    import zlib
    import numpy as np
    import pandas as pd
    import jqdata
    import jqresearch.api as jq_api

    bundle = json.loads(zlib.decompress(base64.b64decode(
        __EXPORT_BUNDLE_LITERAL__
    )).decode("utf-8"))
    plan = bundle["plan"]
    contract = bundle["contract"]
    market_zone = datetime.timezone(datetime.timedelta(hours=8))

    def canonical_bytes(value):
        return json.dumps(value, ensure_ascii=False, sort_keys=True,
                          separators=(",", ":"), allow_nan=False).encode("utf-8")

    def canonical_sha256(value):
        return hashlib.sha256(canonical_bytes(value)).hexdigest()

    def now_text():
        return datetime.datetime.now(market_zone).isoformat()

    def expected_keys_for_observation(observation):
        trading_date = datetime.datetime.strptime(observation["trading_date"], "%Y-%m-%d")
        if trading_date.strftime("%Y-%m-%d") != observation["trading_date"]:
            raise py_builtins.ValueError("交易日不是 canonical 日期")
        if observation["bar_frequency"] == "1d":
            if (observation["session_number"] != 0
                    or observation["session_start_at"] is not None
                    or observation["session_end_at"] is not None):
                raise py_builtins.ValueError("日线观察块结构错误")
            keys = [observation["trading_date"]]
        else:
            if (py_builtins.type(observation["session_number"]) is not py_builtins.int
                    or not 0 < observation["session_number"] <= 127):
                raise py_builtins.ValueError("分钟 Session 编号错误")
            first = datetime.datetime.strptime(
                observation["session_start_at"], "%Y-%m-%dT%H:%M:%S+08:00")
            last = datetime.datetime.strptime(
                observation["session_end_at"], "%Y-%m-%dT%H:%M:%S+08:00")
            if first.second or last.second or first >= last:
                raise py_builtins.ValueError("分钟边界反向或不在整分钟")
            count = py_builtins.int((last - first).total_seconds() / 60)
            if count > contract["max_source_keys"]:
                raise py_builtins.ValueError("单观察块超过来源请求容量")
            keys = [(first + datetime.timedelta(minutes=offset)).strftime(
                "%Y-%m-%dT%H:%M:%S+08:00")
                for offset in py_builtins.range(1, count + 1)]
        if (py_builtins.type(observation["expected_key_count"]) is not py_builtins.int
                or py_builtins.len(keys) != observation["expected_key_count"]
                or canonical_sha256(keys) != observation["expected_keys_sha256"]):
            raise py_builtins.ValueError("理论键数或摘要不匹配")
        ready_at = (trading_date + datetime.timedelta(days=1)).replace(
            hour=contract["ready_hour"], minute=contract["ready_minute"],
            tzinfo=market_zone)
        if datetime.datetime.now(market_zone) < ready_at:
            raise py_builtins.ValueError("计划含尚未就绪的交易日")
        return keys

    def key_summary(keys):
        return {"key_count": py_builtins.len(keys),
                "first_key": keys[0] if keys else None,
                "last_key": keys[-1] if keys else None,
                "keys_sha256": canonical_sha256(keys)}

    # 在任何 API 调用或文件写入前验证整份计划及全部边界。
    if py_builtins.set(plan) != {
            "format_id", "protocol_version", "generator_version",
            "planner_version", "market_time_zone", "source_requests"}:
        raise py_builtins.ValueError("计划顶层字段不匹配")
    for name in ("format_id", "protocol_version", "generator_version",
                 "planner_version", "market_time_zone"):
        if plan[name] != contract[name]:
            raise py_builtins.ValueError("计划版本或标识不匹配: " + name)
    if canonical_sha256(plan) != bundle["plan_sha256"]:
        raise py_builtins.ValueError("完整计划摘要不一致，请重新复制")
    sources = plan["source_requests"]
    if not sources or py_builtins.len(sources) > contract["max_source_requests"]:
        raise py_builtins.ValueError("计划为空或超过来源请求数上限")
    source_ids = py_builtins.set()
    expected_by_request = {}
    source_ranges = {}
    observation_ordinal = 0
    total_expected_count = 0
    for source_ordinal, source in py_builtins.enumerate(sources, 1):
        if py_builtins.set(source) != {
                "source_request_id", "source_request_ordinal", "bar_frequency",
                "contract_code", "observation_request_ids", "source_end_at",
                "source_fields", "source_frequency", "source_parameters",
                "source_start_at", "observations"}:
            raise py_builtins.ValueError("来源请求字段错误")
        source_identity = {key: value for key, value in source.items()
                           if key not in ("source_request_id", "source_request_ordinal", "observations")}
        source_id = source["source_request_id"]
        if (source_id in source_ids or canonical_sha256(source_identity) != source_id
                or source["source_request_ordinal"] != source_ordinal):
            raise py_builtins.ValueError("来源请求身份、序号或唯一性错误")
        source_ids.add(source_id)
        match = re.fullmatch(r"([A-Z]+)([0-9]{4})\.CCFX", source["contract_code"])
        if (match is None or match.group(1) not in contract["underlying_codes"]
                or match.group(2) in ("8888", "9998", "9999")):
            raise py_builtins.ValueError("非白名单固定月份合约")
        frequency = source["bar_frequency"]
        if frequency not in ("1d", "1m"):
            raise py_builtins.ValueError("不支持的频率")
        if (source["source_fields"] != contract["source_fields"][frequency]
                or source["source_frequency"] != ("daily" if frequency == "1d" else "1m")
                or source["source_parameters"] != contract["source_parameters"]):
            raise py_builtins.ValueError("来源字段、频率或参数错误")
        if not source["observations"]:
            raise py_builtins.ValueError("来源请求没有观察块")
        source_keys = []
        observation_ids = []
        for observation in source["observations"]:
            observation_ordinal += 1
            if py_builtins.set(observation) != {
                    "request_id", "request_ordinal", "bar_frequency", "contract_code",
                    "expected_key_count", "expected_keys_sha256", "session_end_at",
                    "session_number", "session_start_at", "trading_date"}:
                raise py_builtins.ValueError("观察块字段错误")
            identity = {key: value for key, value in observation.items()
                        if key not in ("request_id", "request_ordinal")}
            request_id = observation["request_id"]
            if (request_id in expected_by_request
                    or canonical_sha256(identity) != request_id
                    or observation["request_ordinal"] != observation_ordinal
                    or observation["contract_code"] != source["contract_code"]
                    or observation["bar_frequency"] != frequency):
                raise py_builtins.ValueError("观察身份、序号或来源归属错误")
            keys = expected_keys_for_observation(observation)
            source_keys.extend(keys)
            observation_ids.append(request_id)
            expected_by_request[request_id] = key_summary(keys)
        if (source["observation_request_ids"] != observation_ids
                or source_keys != py_builtins.sorted(py_builtins.set(source_keys))
                or source["source_start_at"] != source_keys[0]
                or source["source_end_at"] != source_keys[-1]
                or py_builtins.len(source_keys) > contract["max_source_keys"]):
            raise py_builtins.ValueError("来源请求边界、键顺序或容量错误")
        range_key = (frequency, source["contract_code"])
        intervals = source_ranges.setdefault(range_key, [])
        for first, last in intervals:
            if source_keys[0] <= last and first <= source_keys[-1]:
                raise py_builtins.ValueError("来源请求范围重叠")
        intervals.append((source_keys[0], source_keys[-1]))
        total_expected_count += py_builtins.len(source_keys)
    if (total_expected_count > contract["max_transfer_keys"]
            or observation_ordinal > contract["max_observations"]):
        raise py_builtins.ValueError("计划超过单文件容量")

    run_id = py_builtins.str(uuid.uuid4())
    run_started_at = now_text()
    request_blocks = []
    daily_records = []
    minute_records = []
    failed_sources = []
    stop_reason = None
    for source in sources:
        started_at = now_text()
        started_clock = time.perf_counter()
        completed_at = None
        elapsed_seconds = 0.0
        request_status = "not_executed" if stop_reason else "error"
        block_rows = []
        staged_daily = []
        staged_minute = []
        expected_owner = {}
        source_error_type = None
        source_error_message = None
        for observation in source["observations"]:
            request_id = observation["request_id"]
            observation_id = hashlib.sha256(
                (run_id + "\n" + request_id).encode("utf-8")).hexdigest()
            block_row = {
                "observation_id": observation_id,
                "source_request_id": source["source_request_id"],
                "source_request_ordinal": source["source_request_ordinal"],
                "request_id": request_id, "request_ordinal": observation["request_ordinal"],
                "contract_code": observation["contract_code"],
                "bar_frequency": observation["bar_frequency"],
                "trading_date": observation["trading_date"],
                "session_number": observation["session_number"],
                "session_start_at": observation["session_start_at"],
                "session_end_at": observation["session_end_at"],
                "source_frequency": source["source_frequency"],
                "source_fields": source["source_fields"],
                "source_parameters": source["source_parameters"],
                "expected": expected_by_request[request_id], "actual": None,
                "request_status": request_status, "response_validation_status": "not_run",
                "coverage_status": None, "observation_eligible": False,
                "quality_status": "failed", "quality_reason": "",
                "request_started_at": started_at, "request_completed_at": started_at,
                "elapsed_seconds": 0.0, "error_type": None, "error_message": None,
            }
            block_rows.append(block_row)
            for key in expected_keys_for_observation(observation):
                expected_owner[key] = block_row
        actual_keys_by_request = {row["request_id"]: [] for row in block_rows}
        null_counts_by_request = {
            row["request_id"]: {field: 0 for field in source["source_fields"]}
            for row in block_rows}
        ohlc_warning_counts = {row["request_id"]: 0 for row in block_rows}
        try:
            if stop_reason:
                raise py_builtins.RuntimeError("未执行：前一个来源请求失败；" + stop_reason)
            start_parameter = source["source_start_at"]
            end_parameter = source["source_end_at"]
            if source["bar_frequency"] == "1m":
                start_parameter = start_parameter[:19].replace("T", " ")
                end_parameter = end_parameter[:19].replace("T", " ")
            with warnings.catch_warnings(record=True) as caught_warnings:
                warnings.simplefilter("always")
                response_df = jq_api.get_price(
                    source["contract_code"], start_date=start_parameter,
                    end_date=end_parameter, frequency=source["source_frequency"],
                    fields=source["source_fields"], **source["source_parameters"])
            completed_at = now_text()
            elapsed_seconds = time.perf_counter() - started_clock
            request_status = "success"
            if (not py_builtins.isinstance(response_df, pd.DataFrame)
                    or py_builtins.list(response_df.columns) != source["source_fields"]):
                raise py_builtins.ValueError("响应不是具有精确来源字段的 DataFrame")
            # 聚宽实测空表使用 Float64Index；仅非空表要求无时区 DatetimeIndex。
            if py_builtins.len(response_df) and (
                    not py_builtins.isinstance(response_df.index, pd.DatetimeIndex)
                    or response_df.index.tz is not None):
                raise py_builtins.ValueError("非空响应索引不是实测的无时区 DatetimeIndex")
            response_keys = []
            for timestamp in response_df.index:
                if pd.isnull(timestamp):
                    raise py_builtins.ValueError("行情索引含 NaT")
                timestamp = pd.Timestamp(timestamp)
                if timestamp.second or timestamp.microsecond or timestamp.nanosecond:
                    raise py_builtins.ValueError("行情键不在整分钟")
                if source["bar_frequency"] == "1d":
                    if timestamp.hour or timestamp.minute:
                        raise py_builtins.ValueError("日线键带有非零时刻")
                    response_keys.append(timestamp.strftime("%Y-%m-%d"))
                else:
                    response_keys.append(timestamp.strftime("%Y-%m-%dT%H:%M:%S+08:00"))
            validation_errors = []
            if response_keys != py_builtins.sorted(py_builtins.set(response_keys)):
                validation_errors.append("响应键重复或不严格升序")
            if caught_warnings:
                validation_errors.append("来源发出警告，不能证明无截顶：" + "; ".join(
                    py_builtins.str(item.message) for item in caught_warnings))
            extra_count = 0
            for key, values in py_builtins.zip(
                    response_keys, response_df.itertuples(index=False, name=None)):
                if key not in expected_owner:
                    extra_count += 1
                    continue
                block = expected_owner[key]
                request_id = block["request_id"]
                actual_keys_by_request[request_id].append(key)
                parsed_values = {}
                row_errors = []
                for field, value in py_builtins.zip(source["source_fields"], values):
                    if py_builtins.isinstance(value, np.generic):
                        value = value.item()
                    target_field = "previous_close" if field == "pre_close" else field
                    if value is None or pd.isnull(value):
                        null_counts_by_request[request_id][field] += 1
                        if field == "pre_close":
                            parsed_values[target_field] = None
                        else:
                            row_errors.append(field + " 缺失")
                        continue
                    if (py_builtins.isinstance(value, py_builtins.bool)
                            or not py_builtins.isinstance(value, numbers.Real)):
                        row_errors.append(field + " 类型错误")
                        continue
                    value = py_builtins.float(value)
                    if (not math.isfinite(value)
                            or (field in ("volume", "money", "open_interest") and value < 0)):
                        row_errors.append(field + " 非有限值或负数量")
                        continue
                    parsed_values[target_field] = value
                if row_errors:
                    if py_builtins.len(validation_errors) < 10:
                        validation_errors.append(key + ": " + ", ".join(row_errors))
                    continue
                if (parsed_values["high"] < py_builtins.max(
                        parsed_values["open"], parsed_values["low"], parsed_values["close"])
                        or parsed_values["low"] > py_builtins.min(
                            parsed_values["open"], parsed_values["high"], parsed_values["close"])):
                    ohlc_warning_counts[request_id] += 1
                record = {"observation_id": block["observation_id"],
                          "contract_code": block["contract_code"],
                          "trading_date": block["trading_date"]}
                record.update(parsed_values)
                if source["bar_frequency"] == "1d":
                    staged_daily.append(record)
                else:
                    record["bar_at"] = key
                    record["session_number"] = block["session_number"]
                    staged_minute.append(record)
            if extra_count:
                validation_errors.append("来源响应含计划外键：" + py_builtins.str(extra_count))
            for block in block_rows:
                request_id = block["request_id"]
                keys = actual_keys_by_request[request_id]
                actual = key_summary(keys)
                actual.update({
                    "duplicate_key_count": py_builtins.len(keys) - py_builtins.len(py_builtins.set(keys)),
                    "missing_key_count": block["expected"]["key_count"] - py_builtins.len(py_builtins.set(keys)),
                    "extra_key_count": 0,
                    "null_count_by_field": null_counts_by_request[request_id],
                })
                block["actual"] = actual
            if validation_errors:
                raise py_builtins.ValueError("; ".join(validation_errors))
            for block in block_rows:
                actual_count = block["actual"]["key_count"]
                expected_count = block["expected"]["key_count"]
                coverage = ("complete" if actual_count == expected_count
                            else "empty" if actual_count == 0 else "partial_missing")
                ohlc_warning_count = ohlc_warning_counts[block["request_id"]]
                block.update({
                    "coverage_status": coverage, "observation_eligible": True,
                    "response_validation_status": "passed",
                    "quality_status": "warning" if coverage != "complete" or ohlc_warning_count else "passed",
                    "quality_reason": "coverage=" + coverage + "; ohlc_warning_count=" + py_builtins.str(ohlc_warning_count),
                })
            daily_records.extend(staged_daily)
            minute_records.extend(staged_minute)
        except py_builtins.Exception as error:
            source_error_type = "NotExecuted" if stop_reason else py_builtins.type(error).__name__
            source_error_message = py_builtins.str(error)
            if stop_reason is None:
                stop_reason = source_error_type + ": " + source_error_message
            failed_sources.append({
                "source_request_id": source["source_request_id"],
                "source_request_ordinal": source["source_request_ordinal"],
                "error_type": source_error_type, "error_message": source_error_message,
            })
            for block in block_rows:
                block.update({
                    "observation_eligible": False, "coverage_status": None,
                    "quality_status": "failed", "quality_reason": source_error_message,
                    "response_validation_status": "failed" if request_status == "success" else "not_run",
                    "error_type": source_error_type, "error_message": source_error_message,
                })
        if completed_at is None:
            completed_at = now_text()
            elapsed_seconds = (time.perf_counter() - started_clock
                               if request_status != "not_executed" else 0.0)
        for block in block_rows:
            block.update({"request_status": request_status,
                          "request_started_at": started_at, "request_completed_at": completed_at,
                          "elapsed_seconds": elapsed_seconds})
        request_blocks.extend(block_rows)
        py_builtins.print("source {}/{} {} {}: {}".format(
            source["source_request_ordinal"], py_builtins.len(sources), source["contract_code"],
            source["bar_frequency"], source_error_type or "success"), flush=True)

    successful_source_count = py_builtins.len(sources) - py_builtins.len(failed_sources)
    if successful_source_count == 0:
        raise py_builtins.RuntimeError(
            "没有可提交的成功来源请求；未触碰旧固定文件，不要下载旧文件。首个错误：" + stop_reason)
    daily_records.sort(key=lambda row: (row["contract_code"], row["trading_date"]))
    minute_records.sort(key=lambda row: (row["contract_code"], row["bar_at"]))
    failed_requests = [{
        "source_request_id": row["source_request_id"], "request_id": row["request_id"],
        "error_type": row["error_type"], "error_message": row["error_message"]
    } for row in request_blocks if not row["observation_eligible"]]

    # 独立复核观察与行情成员的键、引用、数量和摘要，再允许序列化与写文件。
    block_by_observation = {row["observation_id"]: row for row in request_blocks}
    if py_builtins.len(block_by_observation) != py_builtins.len(request_blocks):
        raise py_builtins.ValueError("重复 observation_id")
    record_keys_by_observation = {key: [] for key in block_by_observation}
    for frequency, records, field_names in (
            ("1d", daily_records, contract["daily_record_keys"]),
            ("1m", minute_records, contract["minute_record_keys"])):
        previous_key = None
        for row in records:
            if py_builtins.set(row) != py_builtins.set(field_names):
                raise py_builtins.ValueError("行情成员字段不匹配")
            block = block_by_observation[row["observation_id"]]
            key = row["trading_date"] if frequency == "1d" else row["bar_at"]
            compound_key = (row["contract_code"], key)
            if previous_key is not None and compound_key <= previous_key:
                raise py_builtins.ValueError("行情成员全局重复键或排序错误")
            previous_key = compound_key
            if (not block["observation_eligible"] or block["bar_frequency"] != frequency
                    or row["contract_code"] != block["contract_code"]
                    or row["trading_date"] != block["trading_date"]
                    or (frequency == "1m" and row["session_number"] != block["session_number"])):
                raise py_builtins.ValueError("行情成员观察引用错误")
            record_keys_by_observation[block["observation_id"]].append(key)
    for block in request_blocks:
        if py_builtins.set(block) != py_builtins.set(contract["request_block_keys"]):
            raise py_builtins.ValueError("观察成员字段不匹配")
        if block["expected"] != expected_by_request[block["request_id"]]:
            raise py_builtins.ValueError("观察理论键证据与计划不一致")
        keys = record_keys_by_observation[block["observation_id"]]
        if block["observation_eligible"]:
            if (block["request_status"] != "success" or block["response_validation_status"] != "passed"
                    or key_summary(keys) != {key: block["actual"][key] for key in key_summary(keys)}
                    or block["actual"]["duplicate_key_count"] != 0
                    or block["actual"]["extra_key_count"] != 0
                    or block["actual"]["missing_key_count"] != block["expected"]["key_count"] - py_builtins.len(keys)):
                raise py_builtins.ValueError("成功观察与行情成员不一致")
        elif keys:
            raise py_builtins.ValueError("失败观察残留行情记录")

    member_records = {
        "request_blocks.jsonl": request_blocks,
        "futures_daily.jsonl": daily_records,
        "futures_minute.jsonl": minute_records,
    }
    payloads = {}
    member_manifest = {}
    for name in contract["member_names"][1:]:
        records = member_records[name]
        payload = b"".join(canonical_bytes(row) + b"\n" for row in records)
        decoded_count = 0
        for original_row, line in py_builtins.zip(records, io.BytesIO(payload)):
            decoded_row = json.loads(line.decode("utf-8"))
            if decoded_row != original_row or canonical_bytes(decoded_row) + b"\n" != line:
                raise py_builtins.ValueError("JSONL 编码复读不一致: " + name)
            decoded_count += 1
        if decoded_count != py_builtins.len(records):
            raise py_builtins.ValueError("JSONL 复读记录数不一致: " + name)
        payloads[name] = payload
        member_manifest[name] = {"bytes": py_builtins.len(payload),
                                 "sha256": hashlib.sha256(payload).hexdigest(),
                                 "record_count": py_builtins.len(records)}
    manifest = {
        "format_id": contract["format_id"], "protocol_version": contract["protocol_version"],
        "fixed_file_name": contract["fixed_file_name"], "generator_version": contract["generator_version"],
        "planner_version": contract["planner_version"], "plan_sha256": bundle["plan_sha256"],
        "run_id": run_id, "run_started_at": run_started_at, "run_completed_at": now_text(),
        "market_time_zone": contract["market_time_zone"], "source_api_module": contract["source_api_module"],
        "environment": {"python_version": platform.python_version(),
                        "pandas_version": pd.__version__, "numpy_version": np.__version__,
                        "jqdata_version": py_builtins.str(py_builtins.getattr(jqdata, "__version__", "unknown"))},
        "publication_status": "partial" if failed_sources else "complete",
        "source_request_count": py_builtins.len(sources),
        "successful_source_request_count": successful_source_count,
        "failed_source_request_count": py_builtins.len(failed_sources),
        "planned_request_count": py_builtins.len(request_blocks),
        "successful_request_count": py_builtins.len(request_blocks) - py_builtins.len(failed_requests),
        "failed_request_count": py_builtins.len(failed_requests),
        "daily_record_count": py_builtins.len(daily_records), "minute_record_count": py_builtins.len(minute_records),
        "failed_source_requests": failed_sources, "failed_requests": failed_requests,
        "members": member_manifest,
    }
    if py_builtins.set(manifest) != py_builtins.set(contract["manifest_keys"]):
        raise py_builtins.ValueError("Manifest 字段不匹配")
    manifest_payload = canonical_bytes(manifest)
    if canonical_bytes(json.loads(manifest_payload.decode("utf-8"))) != manifest_payload:
        raise py_builtins.ValueError("Manifest 编码复读不一致")
    payloads["manifest.json"] = manifest_payload
    package_buffer = io.BytesIO()
    with zipfile.ZipFile(package_buffer, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for name in contract["member_names"]:
            archive.writestr(name, payloads[name])
    package_bytes = package_buffer.getvalue()

    def validate_package_bytes(value):
        with zipfile.ZipFile(io.BytesIO(value), "r") as archive:
            if (archive.namelist() != contract["member_names"]
                    or archive.testzip() is not None):
                raise py_builtins.ValueError("ZIP 成员或 CRC 错误")
            for info in archive.infolist():
                if (info.compress_type != zipfile.ZIP_DEFLATED or info.flag_bits & 1
                        or archive.read(info.filename) != payloads[info.filename]):
                    raise py_builtins.ValueError("ZIP 成员字节、压缩或加密状态错误")

    validate_package_bytes(package_bytes)
    package_sha256 = hashlib.sha256(package_bytes).hexdigest()
    fixed_path = os.path.abspath(contract["fixed_file_name"])
    if os.path.islink(fixed_path) or (os.path.exists(fixed_path) and not os.path.isfile(fixed_path)):
        raise py_builtins.RuntimeError("固定输出路径不是普通文件，拒绝覆盖")
    temporary_path = None
    try:
        descriptor, temporary_path = tempfile.mkstemp(
            prefix="." + contract["fixed_file_name"] + ".", suffix=".tmp", dir=os.getcwd())
        with os.fdopen(descriptor, "wb") as handle:
            handle.write(package_bytes)
            handle.flush()
            os.fsync(handle.fileno())
        with py_builtins.open(temporary_path, "rb") as handle:
            temporary_bytes = handle.read()
        validate_package_bytes(temporary_bytes)
        if temporary_bytes != package_bytes:
            raise py_builtins.ValueError("临时文件与内存字节不一致")
        os.replace(temporary_path, fixed_path)
        temporary_path = None
        with py_builtins.open(fixed_path, "rb") as handle:
            final_bytes = handle.read()
        validate_package_bytes(final_bytes)
        if final_bytes != package_bytes:
            raise py_builtins.ValueError("固定文件与内存字节不一致")
    finally:
        if temporary_path is not None and os.path.isfile(temporary_path):
            os.remove(temporary_path)
    report = {
        "run_id": run_id, "run_started_at": run_started_at, "published_at": now_text(),
        "plan_sha256": bundle["plan_sha256"], "publication_status": manifest["publication_status"],
        "source_request_count": manifest["source_request_count"],
        "successful_source_request_count": successful_source_count,
        "failed_source_requests": failed_sources,
        "successful_observation_count": manifest["successful_request_count"],
        "failed_observation_count": manifest["failed_request_count"],
        "daily_record_count": py_builtins.len(daily_records),
        "minute_record_count": py_builtins.len(minute_records),
        "download_instruction": {
            "file_name": contract["fixed_file_name"], "file_path": fixed_path,
            "expected_bytes": py_builtins.len(final_bytes), "expected_sha256": package_sha256,
            "action": "手动下载该文件并覆盖本地 data/inbox/ 下同名文件；保留本段完整输出。",
        },
    }
    py_builtins.print("JQ_FINANCIAL_FUTURES_EXPORT_V1_BEGIN")
    py_builtins.print(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False))
    py_builtins.print("JQ_FINANCIAL_FUTURES_EXPORT_V1_END")

_run_jq_financial_futures_export()
'''


## 10．本轮结论或待执行的聚宽代码

这是最后一个单元格。有待办时直接给出当前批次的完整代码和剩余轮数；下载后再次 Run All 即可自动推进。空计划会区分行情齐全、来源确认缺失和上游日历未更新，并给出下一次运行时间，不再统一显示含混的“没有待拉取数据”。

只有聚宽输出完整 END 标记和下载指令后，才下载固定 ZIP；失败或中断时不把旧文件当成本轮产物。


In [11]:
if not round_ready_for_planning or fetch_plan is None:
    raise RuntimeError("本轮尚未完成入库检查和规划，请从头运行 Notebook。")
current_plan_bytes = json.dumps(fetch_plan.plan_payload, ensure_ascii=False, sort_keys=True,
                                separators=(",", ":"), allow_nan=False).encode("utf-8")
if hashlib.sha256(current_plan_bytes).hexdigest() != fetch_plan.plan_sha256:
    raise RuntimeError("内存计划已变化，请重新从头运行，不要生成混合状态的代码。")
if not fetch_plan.plan_payload["source_requests"]:
    missing_key_count = int(fetch_plan.coverage_summary_df["data_missing_key_count"].sum())
    confirmed_missing_key_count = int(fetch_plan.coverage_summary_df["source_confirmed_missing_key_count"].sum())
    if fetch_plan.calendar_is_current and missing_key_count == 0:
        completion_text = "已完成：白名单数据库的日线和分钟行情已全量覆盖当前就绪范围。"
    elif fetch_plan.calendar_is_current:
        completion_text = ("当前可拉取任务已完成，但仍有 " + str(confirmed_missing_key_count)
                           + " 个来源确认缺失键；不是全部行情齐全，当前规则不自动重拉。")
    else:
        completion_text = ("本地已知范围没有普通待办，但上游日历尚未更新到应有日期，不能判定已全量更新到最新。"
                           + " 来源确认缺失键数：" + str(confirmed_missing_key_count) + "。")
    next_run_text = ("可于明天 " + fetch_plan.next_run_at.strftime("%Y-%m-%d %H:%M:%S")
                     + "（北京时间）之后再次从头运行；请确保上游本地日历同步更新。")
    joinquant_code = "# " + completion_text + "\n# " + next_run_text + "\n"
    print(completion_text)
    print("本地日历已知最新应有交易日:", fetch_plan.latest_due_market_trading_date)
    print("自然日日历水位:", fetch_plan.trade_calendar_through, "；当前应覆盖至:", fetch_plan.latest_due_trading_date)
    print("预计剩余普通拉取轮数：0")
    print(next_run_text)
else:
    export_bundle = {
        "plan": fetch_plan.plan_payload,
        "plan_sha256": fetch_plan.plan_sha256,
        "contract": {
            "format_id": local_contract.TRANSFER_FORMAT_ID,
            "protocol_version": local_contract.TRANSFER_PROTOCOL_VERSION,
            "generator_version": local_contract.TRANSFER_GENERATOR_VERSION,
            "planner_version": FETCH_PLANNER_VERSION,
            "market_time_zone": local_contract.TRANSFER_MARKET_TIME_ZONE,
            "source_api_module": local_contract.TRANSFER_SOURCE_API_MODULE,
            "fixed_file_name": local_contract.TRANSFER_FIXED_FILE_NAME,
            "member_names": list(local_contract.TRANSFER_MEMBER_NAMES),
            "manifest_keys": list(local_contract.TRANSFER_MANIFEST_KEYS),
            "request_block_keys": list(local_contract.TRANSFER_REQUEST_BLOCK_KEYS),
            "daily_record_keys": list(local_contract.TRANSFER_DAILY_RECORD_KEYS),
            "minute_record_keys": list(local_contract.TRANSFER_MINUTE_RECORD_KEYS),
            "source_parameters": dict(local_contract.TRANSFER_SOURCE_PARAMETERS),
            "source_fields": {"1d": list(local_contract.JOINQUANT_DAILY_SOURCE_FIELDS),
                              "1m": list(local_contract.JOINQUANT_MINUTE_SOURCE_FIELDS)},
            "underlying_codes": sorted(collection_policy.FINANCIAL_FUTURES_UNDERLYING_CODES),
            "ready_hour": collection_policy.JOINQUANT_COMPLETED_DATA_READY_TIME.hour,
            "ready_minute": collection_policy.JOINQUANT_COMPLETED_DATA_READY_TIME.minute,
            "max_source_keys": collection_policy.JOINQUANT_MAX_EXPECTED_KEYS_PER_SOURCE_REQUEST,
            "max_transfer_keys": collection_policy.JOINQUANT_MAX_EXPECTED_KEYS_PER_TRANSFER,
            "max_observations": collection_policy.JOINQUANT_MAX_OBSERVATIONS_PER_TRANSFER,
            "max_source_requests": collection_policy.JOINQUANT_MAX_SOURCE_REQUESTS_PER_TRANSFER,
        },
    }
    export_bundle_bytes = json.dumps(export_bundle, ensure_ascii=False, sort_keys=True,
                                     separators=(",", ":"), allow_nan=False).encode("utf-8")
    encoded_bundle = base64.b64encode(zlib.compress(export_bundle_bytes, level=9)).decode("ascii")
    bundle_literal = "\n        ".join(repr(encoded_bundle[offset:offset + 100])
                                     for offset in range(0, len(encoded_bundle), 100))
    if joinquant_export_runtime.count("__EXPORT_BUNDLE_LITERAL__") != 1:
        raise RuntimeError("导出模板占位符数量不正确。")
    joinquant_code = joinquant_export_runtime.replace("__EXPORT_BUNDLE_LITERAL__", bundle_literal)
    compile(joinquant_code, "<joinquant-export>", "exec")  # 仅语法检查，不执行。


if fetch_plan.plan_payload["source_requests"]:
    print("预计还需完成", fetch_plan.remaining_transfer_count, "轮拉取（包含下方本轮代码）。")
    print("本轮成功消化后预计还剩", max(0, fetch_plan.remaining_transfer_count - 1), "轮。")
    print("待办 API 请求合计", fetch_plan.remaining_source_request_count, "次；本轮", len(fetch_plan.planned_source_requests_df), "次。")
    print("轮数按当前本地日历快照和全部请求成功估计；每次下载后从头运行会自动重算。")
    if not fetch_plan.calendar_is_current:
        print("上游日历水位尚未跟上：自然日日历仅至", fetch_plan.trade_calendar_through,
              "，当前应覆盖至", fetch_plan.latest_due_trading_date,
              "；交易日历已知最新交易日", fetch_plan.latest_due_market_trading_date,
              "。这些轮数只覆盖当前已知范围，须先同步上游日历才能判断最新状态。")
    print("计划 SHA-256:", fetch_plan.plan_sha256)
    print("完整代码字符数:", len(joinquant_code))
    print("在下方文本框内部 Ctrl+A、Ctrl+C。完整字符串也保留在 joinquant_code 变量中。")
    code_box_html = (
        '<textarea readonly spellcheck="false" rows="24" '
        'style="width:100%;font-family:monospace;color:CanvasText;background:Canvas;'
        'color-scheme:light dark;box-sizing:border-box;" aria-label="完整聚宽代码">'
        + html.escape(joinquant_code) + '</textarea>'
    )
    display({"text/html": code_box_html, "text/plain": joinquant_code}, raw=True)


本地已知范围没有普通待办，但上游日历尚未更新到应有日期，不能判定已全量更新到最新。 来源确认缺失键数：144714。
本地日历已知最新应有交易日: 2026-08-28
自然日日历水位: 2026-08-28 ；当前应覆盖至: 2026-09-04
预计剩余普通拉取轮数：0
可于明天 2026-09-06 00:01:00（北京时间）之后再次从头运行；请确保上游本地日历同步更新。
